## GPR on WeatherBench Data 
Apply GP regression baseline


In [1]:
# Check GPU
!nvidia-smi

Fri Feb 21 14:32:21 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.14              Driver Version: 550.54.14      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla V100-PCIE-16GB           Off |   00000000:1A:00.0 Off |                    0 |
| N/A   34C    P0             27W /  250W |       0MiB /  16384MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr
import seaborn as sns
import pickle
import time
from tqdm.notebook import tqdm

import torch
from torch.utils.data import TensorDataset, DataLoader

import gpytorch
from gpytorch.models import ApproximateGP
from gpytorch.variational import CholeskyVariationalDistribution, VariationalStrategy

import geometric_kernels
import geometric_kernels.torch 
from geometric_kernels.spaces import Hypersphere
from geometric_kernels.kernels import MaternGeometricKernel
from geometric_kernels.frontends.gpytorch import GPyTorchGeometricKernel

INFO (geometric_kernels): Numpy backend is enabled. To enable other backends, don't forget to `import geometric_kernels.*backend name*`.
INFO (geometric_kernels): We may be suppressing some logging of external libraries. To override the logging policy, call `logging.basicConfig`.
INFO (geometric_kernels): Torch backend enabled.


In [3]:
torch.set_default_dtype(torch.float64)

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


A problem with GeometricKernel computations leading to tensors not on same device (check geometric_kernels.kernels.karhunen_loeve line 141)

In [5]:
import geometric_kernels.kernels.karhunen_loeve as kl
import geometric_kernels.spaces.hypersphere as spaces

def new_spectrum(s, nu, lengthscale, dimension):

    assert lengthscale.shape == (1,)
    assert nu.shape == (1,)

    s_tensor = torch.as_tensor(s, dtype=lengthscale.dtype, device=lengthscale.device)
    # Replace np.r_[1.0] with a torch tensor on the correct device
    one_tensor = torch.tensor([1.0], dtype=lengthscale.dtype, device=lengthscale.device)
    # Compute safe_nu: if nu == inf, use 1.0; otherwise keep nu
    safe_nu = torch.where(nu == np.inf, one_tensor, nu)
    # For nu == inf: compute spectral values
    spectral_values_nu_infinite = torch.exp(- (lengthscale**2) / 2.0 * s_tensor).to(device)
    
    # For nu < inf: compute spectral values
    power = -safe_nu - dimension / 2.0
    base = 2.0 * safe_nu / (lengthscale**2) + s_tensor
    spectral_values_nu_finite = base ** power
    
    return torch.where(nu == np.inf, spectral_values_nu_infinite, spectral_values_nu_finite)


def patched_addition_theorem(self, X, X2=None, **kwargs):
    # Determine target device from input X (assumes X is a tensor)
    device = X.device if hasattr(X, "device") else torch.device("cuda")
    # Compute the values and force them to the target device
    values = [
        level.addition(X, X2)[..., None].to(device)  # [N, N2, 1]
        for level in self._spherical_harmonics.harmonic_levels
    ]
    # Concatenate along the last dimension; using torch.cat here
    return torch.cat(values, dim=-1)  # [N, N2, L]

# Apply the monkey patch
kl.MaternKarhunenLoeveKernel.spectrum = staticmethod(new_spectrum)
spaces.SphericalHarmonics._addition_theorem = patched_addition_theorem

In [6]:
def to_pickle(obj, fn):
    with open(fn, 'wb') as f:
        pickle.dump(obj, f)
def read_pickle(fn):
    with open(fn, 'rb') as f:
        return pickle.load(f)

In [6]:
def compute_weighted_mae(da_fc, da_true, mean_dims=xr.ALL_DIMS):
    """ Stolen from WeatherBench score.py
    Compute the MAE with latitude weighting from two xr.DataArrays.
    Args:
        da_fc (xr.DataArray): Forecast. Time coordinate must be validation time.
        da_true (xr.DataArray): Truth.
        mean_dims: dimensions over which to average score
    Returns:
        mae: Latitude weighted root mean absolute error
    """
    error = da_fc - da_true
    weights_lat = np.cos(np.deg2rad(error.lat))
    weights_lat /= weights_lat.mean()
    mae = (np.abs(error) * weights_lat).mean(mean_dims)
    return mae

def compute_weighted_rmse(da_fc, da_true, mean_dims=xr.ALL_DIMS):
    """ Stolen from WeatherBench score.py
    Compute the RMSE with latitude weighting from two xr.DataArrays.

    Args:
        da_fc (xr.DataArray): Forecast. Time coordinate must be validation time.
        da_true (xr.DataArray): Truth.
        mean_dims: dimensions over which to average score
    Returns:
        rmse: Latitude weighted root mean squared error
    """
    error = da_fc - da_true
    weights_lat = np.cos(np.deg2rad(error.lat))
    weights_lat /= weights_lat.mean()
    rmse = np.sqrt(((error)**2 * weights_lat).mean(mean_dims))
    return rmse

## Data Processing

Loading the data

In [21]:
z500 = xr.open_mfdataset('data/5.625deg/geopotential_500/*.nc', combine='by_coords')
z500_train = z500.sel(time=slice('2016', '2017'))['z']
z500_test = z500.sel(time=slice('2018.1', '2018.2'))['z'] # 2017-2018 data

In [22]:
data_mean = z500_train.mean().load()
data_std = z500_train.std().load()
print(data_mean)
print(data_std)

<xarray.DataArray 'z' ()> Size: 4B
array(54213.28, dtype=float32)
Coordinates:
    level    int32 4B 500
<xarray.DataArray 'z' ()> Size: 4B
array(3395.783, dtype=float32)
Coordinates:
    level    int32 4B 500


In [23]:
# Normalize datasets
data_train = (z500_train - data_mean) / data_std
data_test = (z500_test - data_mean) / data_std

In [10]:
"""latitudes = z500_train.lat.values  # Use the dataset's latitude values
longitudes = z500_train.lon.values  # Use the dataset's longitude values
nlat = latitudes.shape[0]
nlon = longitudes.shape[0]
(nlat, nlon)"""

"latitudes = z500_train.lat.values  # Use the dataset's latitude values\nlongitudes = z500_train.lon.values  # Use the dataset's longitude values\nnlat = latitudes.shape[0]\nnlon = longitudes.shape[0]\n(nlat, nlon)"

We need to convert (lat., lon.) coordinates into (x,y,z) for the inputs into the spatial kernel from GeometricKernel 

In [9]:
def latlon_to_cartesian(lat, lon, device=device):
    """ Converting (lat., lon.) to cartesian coordinates on a unit sphere
        Note: WeatherBench data is defined on a constant altitude """

    lat, lon = np.deg2rad(lat), np.deg2rad(lon)

    x = np.cos(lat)[:, None] * np.cos(lon)[None, :]
    y = np.cos(lat)[:, None] * np.sin(lon)[None, :]
    z = np.sin(lat)[:, None] * np.ones_like(lon)[None, :]

    xyz = np.stack([x, y, z], axis=-1)  # Out: (nlat, nlon, 3)
    return torch.tensor(xyz).to(device)

In [12]:
"""cartesian_coords = latlon_to_cartesian(latitudes, longitudes, device) # Shape of data matters
print("Shape of spatial coords: ", cartesian_coords.shape) """

'cartesian_coords = latlon_to_cartesian(latitudes, longitudes, device) # Shape of data matters\nprint("Shape of spatial coords: ", cartesian_coords.shape) '

We also need the time coordinates for the Kernel

In [13]:
"""# Time coordinates 
train_time_coords = (z500_train.time - z500_train.time[0]).values / np.timedelta64(1, 'h')
train_time_coords = torch.tensor(train_time_coords).to(device)
print("Shape of train time coords (units of time): ", train_time_coords.shape)
test_time_coords = (z500_test.time - z500_train.time[0]).values / np.timedelta64(1, 'h')
test_time_coords = torch.tensor(test_time_coords).to(device)
print("Shape of train time coords (units of time): ", test_time_coords.shape)"""

'# Time coordinates \ntrain_time_coords = (z500_train.time - z500_train.time[0]).values / np.timedelta64(1, \'h\')\ntrain_time_coords = torch.tensor(train_time_coords).to(device)\nprint("Shape of train time coords (units of time): ", train_time_coords.shape)\ntest_time_coords = (z500_test.time - z500_train.time[0]).values / np.timedelta64(1, \'h\')\ntest_time_coords = torch.tensor(test_time_coords).to(device)\nprint("Shape of train time coords (units of time): ", test_time_coords.shape)'

In [14]:
"""# Subsample=5 for the data points (same as in LR baseline)
space_subsample = 5
cartesian_coords = cartesian_coords[::space_subsample, ::space_subsample, :]
nlat, nlon, _ = cartesian_coords.shape
print("Subsampled (nlat, nlon): ", (nlat, nlon))

# Subsample every 12 hours
time_subsample = 12
train_time_coords = train_time_coords[::time_subsample]
test_time_coords = test_time_coords[::time_subsample]
print("Subsampled time (train, test): ", (train_time_coords.shape[0], test_time_coords.shape[0]))"""

'# Subsample=5 for the data points (same as in LR baseline)\nspace_subsample = 5\ncartesian_coords = cartesian_coords[::space_subsample, ::space_subsample, :]\nnlat, nlon, _ = cartesian_coords.shape\nprint("Subsampled (nlat, nlon): ", (nlat, nlon))\n\n# Subsample every 12 hours\ntime_subsample = 12\ntrain_time_coords = train_time_coords[::time_subsample]\ntest_time_coords = test_time_coords[::time_subsample]\nprint("Subsampled time (train, test): ", (train_time_coords.shape[0], test_time_coords.shape[0]))'

In [15]:
"""# Combine spatial and temporal data
T = train_time_coords.shape[0]
combined_coords = torch.empty((T, nlat, nlon, 4), dtype=cartesian_coords.dtype).to(device)
# Fill in the spatial part first
combined_coords[..., :3] = cartesian_coords.unsqueeze(0).expand(T, -1, -1, -1)
# The final element is time
combined_coords[..., 3] = train_time_coords.view(T, 1, 1).expand(T, nlat, nlon)
size = combined_coords.element_size() * combined_coords.nelement() / (1024**2)
print(f"Shape: {combined_coords.shape}, Mem: {size:.2f} MB")"""

'# Combine spatial and temporal data\nT = train_time_coords.shape[0]\ncombined_coords = torch.empty((T, nlat, nlon, 4), dtype=cartesian_coords.dtype).to(device)\n# Fill in the spatial part first\ncombined_coords[..., :3] = cartesian_coords.unsqueeze(0).expand(T, -1, -1, -1)\n# The final element is time\ncombined_coords[..., 3] = train_time_coords.view(T, 1, 1).expand(T, nlat, nlon)\nsize = combined_coords.element_size() * combined_coords.nelement() / (1024**2)\nprint(f"Shape: {combined_coords.shape}, Mem: {size:.2f} MB")'

In [ ]:
space_subsample = 5
time_subsample = 1
lead_time = 3*24 # 3 day forecast

def create_training_data(data, lead_time_h, space_subsample=space_subsample, time_subsample=time_subsample, return_valid_time=False, device=device):
    """Preparing input and output data. X should be the coordinate input into the kernel, 
    while Y should be the forecast target (shifted lead time)."""

    X = data.isel(time=slice(0, -lead_time_h, time_subsample),
                        lat=slice(0, None, space_subsample),
                        lon=slice(0, None, space_subsample))
    Y = data.isel(time=slice(lead_time_h, None, time_subsample),
                  lat=slice(0, None, space_subsample),
                  lon=slice(0, None, space_subsample))
    valid_time = Y.time

    # Preparing the input coords (should have shape (ntime, nlat, nlon, 4) if we have no subsampling)
    X_time = (X.time - data.time[0]).values / np.timedelta64(1, 'h')
    X_time = torch.tensor(X_time).to(device)
    t_steps = X_time.shape[0]

    latitudes = X.lat.values 
    longitudes = X.lon.values 
    cartesian_coords = latlon_to_cartesian(X.lat.values, X.lon.values, device)
    nlat, nlon, _ = cartesian_coords.shape

    # Combine spatial and temporal data
    combined_coords = torch.empty((t_steps, nlat, nlon, 4)).to(device)
    # Fill in the spatial part first
    combined_coords[..., :3] = cartesian_coords.unsqueeze(0).expand(t_steps, -1, -1, -1)
    # The final element is time
    combined_coords[..., 3] = X_time.view(t_steps, 1, 1).expand(t_steps, nlat, nlon)
    # Flatten combined_coords to get GP input of shape (N, 4)
    combined_coords = combined_coords.view(-1, 4)
    X_size = combined_coords.element_size() * combined_coords.nelement() / (1024**2)
    print(f"Combined coords info --> Shape: {combined_coords.shape}, Mem: {X_size:.2f} MB")

    # Flatten to shape (N,)
    Y = torch.tensor(Y.values).view(-1).to(device)
    Y_size = Y.element_size() * Y.nelement() / (1024**2)
    print(f"Target info --> Shape: {Y.shape}, Mem: {Y_size:.2f} MB")

    if return_valid_time:
        return combined_coords, Y, latitudes, longitudes, valid_time
    else:
        return combined_coords, Y, latitudes, longitudes

## Defining the Kernel

We work on the 2d-hypersphere, modeling the spatial kernel with Geometric_Kernel and time kernel with periodic kernel. 

In [11]:
# Spatial Kernel from Geometric_Kernel package
sphere = Hypersphere(dim=2)
# Matern Kernel
spatial_kernel_gm = MaternGeometricKernel(sphere)
params = spatial_kernel_gm.init_params()
print('params:', params) # Default kernel uses nu = inf
# Set params here, we want to work with Matern-3/2 (or 5/2)
params["lengthscale"] = torch.tensor([0.5]).to(device)
params["nu"] = torch.tensor([3/2]).to(device)
print('params:', params)

params: {'nu': array([inf]), 'lengthscale': array([1.])}
params: {'nu': tensor([1.5000], device='cuda:0'), 'lengthscale': tensor([0.5000], device='cuda:0')}


In [12]:
# Define the GP regression kernel, separable in space and time
# First convert Geometric Kernel to a GPyTorch compatible kernel
spatial_kernel = gpytorch.kernels.ScaleKernel(
                    GPyTorchGeometricKernel(
                        spatial_kernel_gm,
                        nu = params["nu"],
                        lengthscale=params["lengthscale"],
                        trainable_nu=False,
                        active_dims=[0,1,2]
                    )
                ).to(device)
# spatial_kernel.outputscale = 1.0 #Fix the scale of cov function

# matern_kernel = gpytorch.kernels.MaternKernel(nu=1.5, active_dims=[0,1,2])
# matern_kernel.initialize(lengthscale=params["lengthscale"])
# spatial_kernel = gpytorch.kernels.ScaleKernel(matern_kernel).to(device)

# For temporal kernel
time_kernel1 = gpytorch.kernels.PeriodicKernel(period_length=24, active_dims=[3]).to(device) # Daily    
time_kernel2 = gpytorch.kernels.PeriodicKernel(period_length=24*30, active_dims=[3]).to(device) # Monthly
#time_kernel3 = gpytorch.kernels.PeriodicKernel(period_length=24*30*3, active_dims=[3]).to(device) # Quartly
#time_kernel4 = gpytorch.kernels.PeriodicKernel(period_length=24*30, active_dims=[3]).to(device) # Yearly
time_kernel = time_kernel1 + time_kernel2
#time_kernel = gpytorch.kernels.ExponentialKernel()

# Construct the product kernel
kernel = spatial_kernel * time_kernel

In [ ]:
# Approximate GP
"""class GPModel(ApproximateGP):
    def __init__(self, inducing_points):
        # inducing_points: tensor of shape (M, 4)
        variational_distribution = CholeskyVariationalDistribution(inducing_points.size(0))
        variational_strategy = VariationalStrategy(self, inducing_points, variational_distribution, learn_inducing_locations=True)
        super(GPModel, self).__init__(variational_strategy)
        self.mean_module = gpytorch.means.ConstantMean()
        self.covar_module = kernel

    def forward(self, x):
        # x is expected to be of shape (N, 4) (flattened combined coords)
        mean_x = self.mean_module(x)
        covar_x = self.covar_module(x)
        return gpytorch.distributions.MultivariateNormal(mean_x, covar_x)"""

In [ ]:
# Define the GP model as the product of the spatial and temporal kernel
class GPModel(gpytorch.models.ExactGP):
    def __init__(self, train_x, train_y, likelihood):
        super(GPModel, self).__init__(train_x, train_y, likelihood)
        self.mean_module = gpytorch.means.ConstantMean() 
        self.covar_module = kernel

    def forward(self, x):
        mean_x = self.mean_module(x)
        covar_x = self.covar_module(x)
        return gpytorch.distributions.MultivariateNormal(mean_x, covar_x)

## Training and Evaluation

In [ ]:
def train_gp(lead_time_h, data_train, data_test, epochs=20, batch_size=5000, save_model=False, device=device):
    """
    Train a Gaussian Process model using GPyTorch. Can add subsampling the data for efficiency.
    """
    
    # Prepare training data
    X_train, Y_train, latitudes, longitudes = create_training_data(data_train, lead_time_h, device=device)
    X_test, Y_test, _, _ = create_training_data(data_test, lead_time_h, device=device)

    train_dataset = TensorDataset(X_train, Y_train)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    test_dataset = TensorDataset(X_test, Y_test)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    # Inducing points technique
    # inducing_points = X_train[:1000, :]
    # Define likelihood and GP model
    # model = GPModel(inducing_points=inducing_points).to(device)
    model = GPModel()
    likelihood = gpytorch.likelihoods.GaussianLikelihood().to(device)

    # Train model
    model.train()
    likelihood.train()

    optimizer = torch.optim.Adam(model.parameters(), lr=0.1)
    # Exact GP
    mll = gpytorch.mlls.ExactMarginalLogLikelihood(likelihood, model)
    # Variational GP
    # mll = gpytorch.mlls.VariationalELBO(likelihood, model, num_data=X_train.size(0))

    print("Starting Variational GP regression training:")
    # Reset the peak memory statistics before training
    torch.cuda.reset_peak_memory_stats(device)
    start_time = time.time()

    epochs_iter = tqdm(range(epochs), desc="Epoch")
    for i in epochs_iter:
        # Within each iteration, we will go over each minibatch of data
        minibatch_iter = tqdm(train_loader, desc="Minibatch", leave=False)
        for x_batch, y_batch in minibatch_iter:
            optimizer.zero_grad()
            output = model(x_batch)
            loss = -mll(output, y_batch)
            minibatch_iter.set_postfix(loss=loss.item())
            loss.backward()
            optimizer.step()

    # ... during or after your training loop, you can check:
    current_allocated = torch.cuda.memory_allocated(device)
    max_allocated = torch.cuda.max_memory_allocated(device)

    print(f"Current GPU memory allocated: {current_allocated / (1024**2):.2f} MB")
    print(f"Peak GPU memory allocated: {max_allocated / (1024**2):.2f} MB")

    #with tqdm(total=epochs, desc="Training Progress", unit="epoch") as pbar:
    #    for i in range(epochs):
    #        optimizer.zero_grad()
    #        output = model(X_train)
    #        loss = -mll(output, Y_train)
    #        loss.backward()
    #        optimizer.step()
    #        pbar.set_postfix(loss=f"{loss.item():.3f}")
    #        pbar.update(1)

    end_time = time.time()
    print("Training completed")
    time_elapsed = end_time - start_time
    print(f"Training time: {time_elapsed:.2f}s")

    # Save model
    if save_model:
        torch.save(model.state_dict(), "gp_model.pth")
        torch.save(likelihood.state_dict(), "gp_likelihood.pth")
        print("Saved model")

    return model, likelihood, test_loader, latitudes, longitudes

In [ ]:
# Train GP model
model, likelihood, test_loader, latitudes, longitudes = train_gp(lead_time, data_train, data_test)

In [ ]:
nlat = len(latitudes)
nlon = len(longitudes)

# Make Predictions
model.eval()
likelihood.eval()

# Collect predictions from test_loader in batches
preds = []
with torch.no_grad():
    for x_batch, _ in test_loader:
        batch_pred = likelihood(model(x_batch))
        preds.append(batch_pred.mean.cpu())
y_pred_flat = torch.cat(preds, dim=0)  

N_test = y_pred_flat.size(0)
T_test = N_test // (nlat * nlon)
y_pred_mean = y_pred_flat.view(T_test, nlat, nlon).numpy()

test_times = z500_test.time.values[::time_subsample][:T_test]

y_pred_xr = xr.DataArray(y_pred_mean, dims=['time', 'lat', 'lon'], 
                         coords={'time': test_times, 'lat': latitudes, 'lon': longitudes})
rmse = compute_weighted_rmse(y_pred_xr, z500_test)
print(f"Test RMSE: {rmse.values:.3f}")

Test RMSE: 1524.214
